# 05 Risk Engine

Purpose: review risk snapshot metrics, stress scenarios, correlation summaries and approximate risk contribution.

Data source: `data/processed/risk_snapshot.parquet` generated from real MOEX candle returns where available.


## Methodology Summary

Risk metrics use historical daily returns. VaR and CVaR are empirical tail metrics. Risk contribution is a covariance-based approximation. Stress tests are scenario diagnostics, not forecasts.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from russian_markets_lab.data.io import read_processed_dataset, read_processed_metadata

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def load_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    df = read_processed_dataset(name)
    metadata = read_processed_metadata(name)
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    if metadata:
        print('generated_at:', metadata.get('generated_at'))
        print('source:', metadata.get('source'))
    else:
        print('metadata: missing')
    return df, metadata

def show_missing(name: str) -> None:
    print(f'No processed data found for {name}. Run the corresponding CLI pipeline first.')


In [ ]:
risk, metadata = load_dataset('risk_snapshot')
if risk.empty:
    show_missing('risk_snapshot')
else:
    display(risk)


## Portfolio Metrics


In [ ]:
if not risk.empty and {'section', 'metric', 'value'}.issubset(risk.columns):
    display(risk[risk['section'] == 'portfolio'])


## Stress Scenarios


In [ ]:
if not risk.empty and 'section' in risk.columns:
    stress = risk[risk['section'].astype(str).str.startswith('stress')]
    display(stress)


## Risk Contribution And Key Observations

Describe tail risk, drawdown, concentration, scenario sensitivity and whether approximate risk contribution is concentrated in a small number of instruments.


if not risk.empty and 'section' in risk.columns:
    contribution = risk[risk['section'].astype(str) == 'risk_contribution']
    if contribution.empty:
        print('No risk contribution rows are available in the current snapshot.')
    else:
        display(contribution)
        if {'instrument', 'risk_contribution_pct'}.issubset(contribution.columns):
            fig = px.bar(contribution, x='instrument', y='risk_contribution_pct', title='Approximate risk contribution')
            fig.show()


## Limitations And Disclaimer

Historical risk is backward-looking. Scenario shocks are simplified diagnostics and not investment advice. No full margin, liquidation or order book impact model is included.
